# Elasticsearch Search Queries Cheatsheet

> **Part of:** `cheatsheets` repo
> **Topic:** Search queries — match, match_phrase, term, pagination, sorting, fielddata

---

## Table of Contents
1. [match — OR vs AND operator](#1-match--or-vs-and-operator)
2. [match vs match_phrase](#2-match-vs-match_phrase)
3. [Pagination, _source, fields, sorting](#3-pagination-_source-fields-sorting)
4. [Fielddata error](#4-fielddata-error)
5. [match vs term on nested fields](#5-match-vs-term-on-nested-fields)
6. [Query type quick reference](#6-query-type-quick-reference)
7. [Key rules](#7-key-rules)

---

## 1. match — OR vs AND operator

By default `match` uses `OR` — any document containing either word qualifies.
Adding `operator: "and"` requires **both** words to be present.

```json
// OR (default) — returns more, noisier results
GET blogs_fixed2/_search
{
  "query": {
    "match": {
      "content": "open source"
    }
  }
}

// AND — returns fewer, higher-precision results
GET blogs_fixed2/_search
{
  "query": {
    "match": {
      "content": {
        "query": "open source",
        "operator": "and"
      }
    }
  }
}
```

> **Analogy:** `OR` is like a Google search that shows pages mentioning either word.
> `AND` is like using `+open +source` to force both.

---

## 2. match vs match_phrase

`match_phrase` requires words to appear **adjacent and in order**.
"open source" matches — "source is open" does not.

```json
// match — words can appear anywhere in the field
GET blogs_fixed2/_search
{
  "query": {
    "match": {
      "content": "open source"
    }
  }
}

// match_phrase — words must be adjacent and in order
GET blogs_fixed2/_search
{
  "query": {
    "match_phrase": {
      "content": "open source"
    }
  }
}
```

> **Analogy:** `match` asks "do you own a dog and a leash?"
> `match_phrase` asks "are you walking a dog on a leash right now?"

---

## 3. Pagination, `_source`, `fields`, sorting

### Limit returned fields with `_source`

```json
GET blogs_fixed2/_search
{
  "_source": ["title", "publish_date"],
  "query": {
    "match_phrase": { "content": "open source" }
  }
}
```

### Fetch specific `fields` only (more efficient than `_source`)

```json
GET blogs_fixed2/_search
{
  "size": 15,
  "_source": false,
  "fields": ["title"],
  "query": {
    "match": { "content": "open source" }
  }
}
```

> `_source: false` + `fields` skips deserializing the full stored document.
> Use this when you only need one or two fields from large documents.

### Sort by date (most recent first)

```json
GET blogs_fixed2/_search
{
  "_source": ["title", "publish_date"],
  "size": 5,
  "sort": [
    { "publish_date": { "order": "desc" }}
  ],
  "query": {
    "match_phrase": { "content": "open source" }
  }
}
```

> Requires the field to be mapped as `date`. Works natively on `keyword` and numeric fields too.

### Paginate with `from` + `size`

```json
// Page 1 — first 5 results
GET blogs_fixed2/_search
{
  "_source": ["title", "publish_date"],
  "from": 0,
  "size": 5,
  "sort": [{ "publish_date": { "order": "desc" }}],
  "query": {
    "match_phrase": { "content": "open source" }
  }
}

// Page 2 — next 5 results
GET blogs_fixed2/_search
{
  "_source": ["title", "publish_date"],
  "from": 5,
  "size": 5,
  "sort": [{ "publish_date": { "order": "desc" }}],
  "query": {
    "match_phrase": { "content": "open source" }
  }
}
```

> Equivalent to `OFFSET 5 LIMIT 5` in SQL.
>
> ⚠️ Deep pagination (`from: 10000+`) is expensive — Elasticsearch fetches and discards all
> preceding results. Use `search_after` for deep pagination instead.

---

## 4. Fielddata error

```json
{
  "error": {
    "type": "illegal_argument_exception",
    "reason": "Can't load fielddata on [url] because fielddata is unsupported on fields of type [keyword]. Use doc values instead."
  }
}
```

### What it means

`fielddata` is an in-memory inverted structure designed for `text` fields.
`keyword` fields use **doc values** (an efficient column-store) instead.

This error fires when a query or aggregation tries to load fielddata on a `keyword` field.

### Fix

Just reference the `keyword` field directly in your aggregation or sort — doc values are used automatically.

```json
// Wrong — triggers fielddata on a text field
{ "aggs": { "by_url": { "terms": { "field": "url_text_field" }}}}

// Correct — keyword field uses doc values natively
{ "aggs": { "by_url": { "terms": { "field": "url" }}}}
```

---

## 5. match vs term on nested fields

`job_title` is mapped as both `text` and `.keyword` — giving you two different query strategies.

```json
// Count matching docs
GET blogs_fixed2/_count
{
  "query": {
    "match": { "authors.job_title": "Director of Engineering" }
  }
}
```

```json
// match on text field — analyzer runs
// Finds: "Director of Engineering", "Senior Director of Engineering", "Engineering Director"
GET blogs_fixed2/_search
{
  "_source": ["authors.job_title"],
  "query": {
    "match": { "authors.job_title": "Director of Engineering" }
  }
}
```

```json
// term on keyword field — exact match only
// Finds ONLY: "Director of Engineering" (nothing more, nothing less)
GET blogs_fixed2/_search
{
  "query": {
    "term": { "authors.job_title.keyword": "Director of Engineering" }
  }
}
```

```json
// Paginate deep into term results
GET blogs_fixed2/_search
{
  "_source": ["authors"],
  "from": 78,
  "query": {
    "term": { "authors.job_title.keyword": "Director of Engineering" }
  }
}
```

> `from: 78` means there are at least 78 exact matches — use `_count` first to check
> result volume before paginating.

---

## 6. Query type quick reference

| Goal | Field type | Query |
|------|-----------|-------|
| Any of these words | `text` | `match` (default OR) |
| All of these words | `text` | `match` with `operator: "and"` |
| Exact phrase, word order matters | `text` | `match_phrase` |
| Exact single value | `keyword` | `term` |
| Match any of several values | `keyword` | `terms` |
| Sort or aggregate | `keyword` | doc values — works natively |
| Full-text search + sort/agg | multi-field | `match` on `field`, `term`/sort on `field.keyword` |
| Count results without fetching docs | any | `_count` |

---

## 7. Key rules

**Never use `term` on a `text` field.** The analyzer transforms stored tokens at index time, so exact matches almost never work. Always use `term` on `.keyword` subfields.

**Never search both `title` and `title.keyword` in the same query.** You'll get duplicate results with inflated scores. Pick one subfield per query.

**`_count` is fast.** It skips document fetching entirely. Use it before large searches to know your result volume upfront.

**`_source: false` + `fields` is the most efficient retrieval pattern** when you only need one or two fields from large documents.

**`match` with `operator: "and"` ≠ `match_phrase`.** AND requires both words anywhere in the field. `match_phrase` requires them adjacent and in order.

---

*Part of personal cheatsheets repo — github.com/longchung90/cheatsheets*
